# Regresión Lineal con PyTorch — Predicción de Contaminación NO₂ en EE.UU.

**Dataset:** US Pollution 2000-2021  
**Fuente:** https://www.kaggle.com/datasets/alpacanonymous/us-pollution-20002021

## Variable objetivo (Y)
`NO2 Mean` — Concentración promedio diaria de NO₂ en partes por billón (ppb)

## Features seleccionadas (n=14)
| # | Feature | Justificación |
|---|---|---|
| 1 | `CO Mean` | Mismo origen que NO₂ — combustión de motores |
| 2 | `SO2 Mean` | Coexiste con NO₂ en zonas industriales |
| 3 | `O3 Mean` | Relación química directa con NO₂ |
| 4 | `CO 1st Max Value` | Pico extremo del día |
| 5 | `SO2 1st Max Value` | Pico industrial del día |
| 6 | `O3 1st Max Value` | Pico de reacción química |
| 7 | `CO 1st Max Hour` | Hora del pico indica tipo de fuente |
| 8 | `SO2 1st Max Hour` | Hora de actividad industrial |
| 9 | `O3 1st Max Hour` | Hora de mayor reacción química |
| 10 | `Year` | Tendencia histórica de reducción de emisiones |
| 11 | `Month` | Estacionalidad — más NO₂ en invierno |
| 12 | `day_of_week` | Patrón laboral vs fin de semana |
| 13 | `State` | Contexto regional y regulatorio |
| 14 | `City` | Detalle urbano vs rural |

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from matplotlib import pyplot
from sklearn.metrics import r2_score

%matplotlib inline

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Carga y Preprocesamiento con Pandas

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Intelegencia Artificial/Dataset/pollution_2000_2021.csv')

data['Date']        = pd.to_datetime(data['Date'])
data['day_of_week'] = data['Date'].dt.dayofweek

data['State'] = data['State'].astype('category').cat.codes
data['City']  = data['City'].astype('category').cat.codes

FEATURES = [
    'CO Mean', 'SO2 Mean', 'O3 Mean',
    'CO 1st Max Value', 'SO2 1st Max Value', 'O3 1st Max Value',
    'CO 1st Max Hour', 'SO2 1st Max Hour', 'O3 1st Max Hour',
    'Year', 'Month', 'day_of_week', 'State', 'City'
]
TARGET = 'NO2 Mean'

X = data[FEATURES].values.astype(np.float32)
y = data[TARGET].values.astype(np.float32)
m = y.size

print(f'X shape : {X.shape}')
print(f'y shape : {y.shape}')
print(f'm       : {m:,}')

## 2. Normalización y Split 80/20

In [ ]:
# se calcula la media y desviacion estandar para normalizar los datos
mu    = np.mean(X, axis=0)
sigma = np.std(X, axis=0)
sigma[sigma == 0] = 1
X = ((X - mu) / sigma).astype(np.float32)

np.random.seed(42)
indices = np.random.permutation(m)
split   = int(0.8 * m)

X_train = X[indices[:split]]
y_train = y[indices[:split]]
X_test  = X[indices[split:]]
y_test  = y[indices[split:]]

print(f'Entrenamiento: {len(y_train):,}')
print(f'Validación   : {len(y_test):,}')

In [ ]:
# se convierten los arrays de numpy a tensores de pytorch
X_t = torch.from_numpy(X_train).float()
Y_t = torch.from_numpy(y_train).float().view(-1, 1)

print(f'X_t shape: {X_t.shape}')
print(f'Y_t shape: {Y_t.shape}')

## 3. Definir el Modelo

Se crea una clase que hereda de `torch.nn.Module`.
En `__init__` se definen las capas y en `forward` la lógica de cálculo.

In [ ]:
# creamos una clase que hereda de torch.nn.Module
class ModeloRegresionLineal(nn.Module):

    # constructor
    def __init__(self, D_in, D_out):

        # llamamos al constructor de la clase madre
        super(ModeloRegresionLineal, self).__init__()

        # definimos la capa lineal
        self.fc = nn.Linear(D_in, D_out)

    # lógica para calcular las salidas de la red
    def forward(self, x):
        x = self.fc(x)  # sin activacion = regresion lineal
        return x


D_in  = X_t.shape[1]  # 14 features
D_out = 1             # predecir un valor continuo de NO2

model = ModeloRegresionLineal(D_in, D_out)
print(model)

# verificar que el modelo recibe los datos en la forma correcta
x_prueba = torch.randn(5, D_in)
outputs  = model(x_prueba)
print(outputs.shape)

## 4. Función de Pérdida y Optimizador

In [ ]:
criterion = nn.MSELoss()  # Error cuadrático medio
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)  # Descenso de gradiente estocástico

## 5. Entrenamiento

In [ ]:
epochs   = 1000
log_each = 100
l = []
model.train()
for e in range(1, epochs + 1):

    # se calculan las predicciones con los parametros actuales
    y_pred = model(X_t)

    # se calcula el error entre la prediccion y el valor real
    loss = criterion(y_pred, Y_t)
    l.append(loss.item())

    # se limpian los gradientes del paso anterior para no acumularlos
    optimizer.zero_grad()

    # se calculan automaticamente todos los gradientes
    loss.backward()

    # se actualizan los parametros del modelo con los gradientes calculados
    optimizer.step()

    if not e % log_each:
        print(f'Epoch {e}/{epochs} Loss {np.mean(l):.5f}')

## 6. Gráfica de Convergencia del Costo

In [ ]:
pyplot.plot(np.arange(len(l)), l, lw=2)
pyplot.xlabel('Número de iteraciones')
pyplot.ylabel('Costo J')
pass

## 7. Evaluación del Modelo

In [ ]:
def evaluate(x):
    model.eval()
    with torch.no_grad():
        return model(x).squeeze().numpy()

y_pred = evaluate(torch.from_numpy(X_test).float())
print('R² : {:.4f} → el modelo explica el {:.2f}% de la variación del NO₂'.format(
    r2_score(y_test, y_pred), r2_score(y_test, y_pred) * 100))

## 8. Predicciones finales

In [ ]:
XPrueba = torch.from_numpy(X_test[:30]).float()
pred    = evaluate(XPrueba)

print('Predicciones finales:')
for i in range(30):
    print(f'Real: {y_test[i]:.2f} ppb,  Predicción: {pred[i]:.2f} ppb')